In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%pip install -q \
opentelemetry-sdk==1.38.0 \
opentelemetry-proto==1.38.0 \
opentelemetry-exporter-otlp-proto-common==1.38.0 \
  langchain \
  langchain-community \
  langchain-core \
  langchain-groq \
  langgraph \
  chromadb \
  sentence-transformers \
  transformers \
  openai-whisper \
  pillow \
  tqdm \
  requests \
  accelerate \
  pydantic

In [43]:
import json
from tqdm import tqdm
import os
import torch
import numpy as np
import whisper
import requests
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import torch.nn.functional as F
import pickle
from collections import defaultdict
from typing import Optional,List
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import time

In [4]:
def normalize_instagram(r: dict) -> dict:
    return {
        "id":          f"ig_{r.get('id','')}",
        "source":      "instagram",
        "title":       r.get("caption","")[:100],
        "caption":     r.get("caption",""),
        "video_url":   r.get("audioUrl",""),
        "thumbnail":   r.get("displayUrl",""),
        "likes":       r.get("likesCount", 0),
        "comments":    r.get("commentsCount", 0),
        "duration":    r.get("videoDuration", 0),
        "owner":       r.get("ownerUsername",""),
        "hashtags":    r.get("hashtags",[]),
        "timestamp":   r.get("timestamp",""),
        "transcript":  "",
        "thumb_path":  None,
        "rich_text":   "",
    }



# print("📂 Loading raw JSONs...")
with open("/content/drive/MyDrive/Dataset-json/output.json","r",encoding="utf-8") as f:
    ig_data = json.load(f)


merged = []
for r in tqdm(ig_data, desc="📸 Instagram"): merged.append(normalize_instagram(r))

seen, deduped = set(), []
for r in merged:
    key = r.get("video_url") or r.get("id")
    if key not in seen:
        deduped.append(r)
        seen.add(key)

with open("final-dataset1.json", "w", encoding="utf-8") as f:
    json.dump(deduped, f, indent=2, ensure_ascii=False)


📸 Instagram: 100%|██████████| 1140/1140 [00:00<00:00, 192468.97it/s]


In [5]:


# ── Device ────────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")

# ── Models ────────────────────────────────────────────────────────────────────
print("Loading models...")
text_model     = SentenceTransformer("all-MiniLM-L6-v2", device=device)
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
whisper_model  = whisper.load_model("medium", device=device)
clip_model.eval()
print("All models loaded!")

# ── ChromaDB ──────────────────────────────────────────────────────────────────
client           = chromadb.PersistentClient(path="./chroma_db")
text_collection  = client.get_or_create_collection(
    name="reel_text",   metadata={"hnsw:space": "cosine"})
image_collection = client.get_or_create_collection(
    name="reel_images", metadata={"hnsw:space": "cosine"})

# ── Load data ─────────────────────────────────────────────────────────────────
with open("/content/drive/MyDrive/Dataset-json/output.json") as f:
    reels = json.load(f)

os.makedirs("thumbnails", exist_ok=True)
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# ── Helpers ───────────────────────────────────────────────────────────────────
def transcribe(video_url: str, reel_id: str) -> str:
    if not video_url:
        return ""
    tmp = f"tmp_{reel_id}.mp4"
    try:
        r = requests.get(video_url, timeout=30, stream=True, headers=HEADERS)
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=16384):
                f.write(chunk)
        result = whisper_model.transcribe(
            tmp,
            fp16=(device == "cuda"),
            language="en",
            verbose=False,
            condition_on_previous_text=False,
        )
        return result["text"].strip()
    except Exception as e:
        tqdm.write(f"Whisper failed {reel_id}: {e}")
        return ""
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


def download_thumbnail(url: str, save_path: str) -> bool:
    try:
        r   = requests.get(url, timeout=8, headers=HEADERS)
        img = Image.open(BytesIO(r.content)).convert("RGB")
        if img.size[0] < 50 or img.size[1] < 50:
            return False
        img.save(save_path, "JPEG", quality=85)
        return True
    except Exception:
        return False


def get_clip_embedding(image_path: str):
    try:
        image  = Image.open(image_path).convert("RGB")
        inputs = clip_processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            features = clip_model.get_image_features(**inputs)
            features = features / features.norm(dim=-1, keepdim=True)
        return features[0].cpu().numpy().tolist()
    except Exception:
        return None


# ── Main loop ─────────────────────────────────────────────────────────────────
text_ok = text_fail = img_ok = img_fail = 0

for reel in tqdm(reels, desc="Processing reels", unit="reel"):
    reel_id   = str(reel["id"])
    video_url = reel.get("video_url", "")

    # Step 1: Transcribe if no transcript yet
    transcript = reel.get("transcript", "").strip()
    if not transcript and video_url:
        transcript = transcribe(video_url, reel_id)
        reel["transcript"] = transcript

    # Step 2: Download thumbnail if not already on disk
    thumb_path = reel.get("thumb_path")
    if not thumb_path or not os.path.exists(str(thumb_path)):
        thumb_url = reel.get("thumbnail")
        if thumb_url:
            save_path = f"thumbnails/{reel_id}.jpg"
            success   = download_thumbnail(thumb_url, save_path)
            thumb_path = save_path if success else None
            reel["thumb_path"] = thumb_path


    metadata = {
        "video_url": video_url,
        "owner":     reel.get("owner", ""),
        "likes":     reel.get("likes", 0),
        "comments":  reel.get("comments", 0),
        "duration":  reel.get("duration", 0),
        "source":    reel.get("source", "unknown"),
        "caption":   reel.get("caption", "")[:500],
    }

    # Step 3: Text embedding
    full_text = f"{reel.get('caption','')} {transcript}".strip()
    if full_text:
        try:
            text_embed = text_model.encode(
                full_text,
                normalize_embeddings=True
            ).tolist()
            text_collection.upsert(
                ids=[reel_id],
                embeddings=[text_embed],
                documents=[full_text[:2000]],
                metadatas=[metadata]
            )
            text_ok += 1
        except Exception as e:
            tqdm.write(f"Text embed failed {reel_id}: {e}")
            text_fail += 1

    # Step 4: CLIP embedding
    if thumb_path and os.path.exists(thumb_path):
        img_embed = get_clip_embedding(thumb_path)
        if img_embed:
            try:
                image_collection.upsert(
                    ids=[reel_id],
                    embeddings=[img_embed],
                    metadatas=[metadata]
                )
                img_ok += 1
            except Exception as e:
                tqdm.write(f"Image embed failed {reel_id}: {e}")
                img_fail += 1
        else:
            img_fail += 1

# Save back with transcripts filled in
with open("processed.json", "w", encoding="utf-8") as f:
    json.dump(reels, f, indent=2, ensure_ascii=False)

print(f"""
ChromaDB populated!
   Text  -> {text_ok} stored  {text_fail} failed
   Image -> {img_ok} stored  {img_fail} failed
   Text collection  : {text_collection.count()} docs
   Image collection : {image_collection.count()} docs
""")

Device: CPU
Loading models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


All models loaded!


Processing reels: 100%|██████████| 1140/1140 [02:24<00:00,  7.92reel/s]



ChromaDB populated!
   Text  -> 1134 stored  0 failed
   Image -> 0 stored  0 failed
   Text collection  : 1134 docs
   Image collection : 1123 docs



In [42]:
with open("/content/drive/MyDrive/Dataset-json/output.json") as f:
    raw = json.load(f)

raw_lookup = {str(r.get("id","")): r for r in raw}
client    = chromadb.PersistentClient(path="./chroma_db")

for col in [client.get_collection("reel_text"), client.get_collection("reel_images")]:
    total = col.count()
    print(f"Patching {col.name}: {total} docs...")
    for offset in range(0, total, 500):
        result     = col.get(limit=500, offset=offset, include=["metadatas"])
        ids, metas = result["ids"], result["metadatas"]
        for rid, meta in zip(ids, metas):
            r  = raw_lookup.get(rid.replace("ig_",""), {})
            sc = r.get("shortCode", "")
            meta["video_url"] = f"https://www.instagram.com/p/{sc}/"
            meta["embed_url"] = f"https://www.instagram.com/p/{sc}/embed/"
        col.update(ids=ids, metadatas=metas)
    print(f"  Done!")

sample = client.get_collection("reel_text").get(limit=1, include=["metadatas"])
print(f"video_url: {sample['metadatas'][0].get('video_url')}")

Patching reel_text: 1134 docs...
  Done!
Patching reel_images: 1123 docs...
  Done!
video_url: https://www.instagram.com/p/DTQBOwdj7DU/


In [15]:
with open("/content/drive/MyDrive/Dataset-json/output.json") as f:
    data = json.load(f)

print(f"Total records: {len(data)}")
print(f"Thumbnails on disk: {len(os.listdir('thumbnails'))}")

patched = 0
for reel in data:
    reel_id    = str(reel.get("id", ""))
    thumb_path = f"thumbnails/ig_{reel_id}.jpg"   # ← ig_ prefix since normalize adds it
    if os.path.exists(thumb_path):
        reel["thumb_path"] = thumb_path
        patched += 1
    else:
        reel["thumb_path"] = None

print(f"Patched: {patched}")

with open("output.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print("Saved!")

Total records: 1140
Thumbnails on disk: 1123
Patched: 1123
Saved!


In [16]:
with open("output.json") as f:
    data = json.load(f)

device         = "cuda" if torch.cuda.is_available() else "cpu"
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

client    = chromadb.PersistentClient(path="./chroma_db")
image_col = client.get_or_create_collection(
    name="reel_images", metadata={"hnsw:space": "cosine"})

os.makedirs("thumbnails", exist_ok=True)
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
ok = fail = skip = 0

for reel in tqdm(data, desc="CLIP embedding", unit="reel"):
    reel_id    = f"ig_{reel.get('id','')}"
    thumb_url  = reel.get("displayUrl","")
    thumb_path = f"thumbnails/{reel_id}.jpg"

    if not thumb_url:
        skip += 1
        continue

    if not os.path.exists(thumb_path):
        try:
            r   = requests.get(thumb_url, timeout=8, headers=HEADERS)
            img = Image.open(BytesIO(r.content)).convert("RGB")
            if img.size[0] < 50:
                skip += 1
                continue
            img.save(thumb_path, "JPEG", quality=85)
        except Exception as e:
            tqdm.write(f"Download failed {reel_id}: {e}")
            skip += 1
            continue

    try:
        image        = Image.open(thumb_path).convert("RGB")
        inputs       = clip_processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)

        with torch.no_grad():
            out      = clip_model.get_image_features(pixel_values=pixel_values)
            features = F.normalize(out.pooler_output, p=2, dim=-1)

        embed = features[0].cpu().numpy().tolist()

        metadata = {
            "video_url": reel.get("audioUrl", ""),
            "owner":     reel.get("ownerUsername", ""),
            "likes":     reel.get("likesCount", 0),
            "comments":  reel.get("commentsCount", 0),
            "duration":  reel.get("videoDuration", 0),
            "source":    "instagram",
            "caption":   reel.get("caption", "")[:500],
        }

        image_col.upsert(
            ids=[reel_id],
            embeddings=[embed],
            metadatas=[metadata]
        )
        ok += 1

    except Exception as e:
        tqdm.write(f"CLIP failed {reel_id}: {e}")
        fail += 1

print(f"""
Done!
   Embedded : {ok:,}
   Failed   : {fail:,}
   Skipped  : {skip:,}
   Image collection: {image_col.count():,} docs
""")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
CLIP embedding:   3%|▎         | 29/1140 [00:10<06:04,  3.05reel/s]

Download failed ig_3834352259840930613: HTTPSConnectionPool(host='instagram.fcps3-1.fna.fbcdn.net', port=443): Max retries exceeded with url: /v/t51.2885-15/635061677_26342008245393168_2097056988418163884_n.jpg?stp=dst-jpg_e15_tt6&_nc_ht=instagram.fcps3-1.fna.fbcdn.net&_nc_cat=100&_nc_oc=Q6cZ2QGav_Ls7pbp6rnkmqrRsDZf4udjo5fcuVXAow6ibODYofTuozKz-umgxzPCMSSio6z2PXV4NUtR4XcdIH_hpMTj&_nc_ohc=iTEdVOTlTuAQ7kNvwHU2z2o&_nc_gid=IdqBP4H5xMDr7D5hAtKc6w&edm=APs17CUBAAAA&ccb=7-5&oh=00_AftmCZGmBmY7q93uEzL_r1Lzc1PI_PQ726_88OmGn3_WRQ&oe=699E33A3&_nc_sid=10d13b (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x79c1d3f5a630>: Failed to establish a new connection: [Errno 101] Network is unreachable'))


CLIP embedding:   9%|▉         | 100/1140 [00:30<04:39,  3.73reel/s]

Download failed ig_3613248286225087903: HTTPSConnectionPool(host='instagram.fosu2-1.fna.fbcdn.net', port=443): Max retries exceeded with url: /v/t51.2885-15/544898925_18293554378254861_7852618799278793821_n.jpg?stp=dst-jpg_e15_tt6&_nc_ht=instagram.fosu2-1.fna.fbcdn.net&_nc_cat=109&_nc_oc=Q6cZ2QGnL1fqx1B7Cot_vKw1YhCKCxaNiZ7fyBfX3LQ1F5gN6QlmUm7tJoeEtukq-Cz8o2cBCZiKiiPXEXAAV-j4RdYv&_nc_ohc=lMUJMwr7BgwQ7kNvwH4QF_H&_nc_gid=CWvtKLNuc_14fw5dsT7hHg&edm=APs17CUBAAAA&ccb=7-5&oh=00_AfsIJ272OmzH_ifq0NIaBbDyErU6CEEwq4_zajjOVAaOpw&oe=699E5255&_nc_sid=10d13b (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x79c29c2840b0>: Failed to establish a new connection: [Errno 101] Network is unreachable'))


...

Download failed ig_3106950881376028817: HTTPSConnectionPool(host='instagram.ftpa1-2.fna.fbcdn.net', port=443): Max retries exceeded with url: /v/t51.2885-15/501959825_733827835966074_8745568792406746809_n.jpg?stp=dst-jpg_e15_tt6&_nc_ht=instagram.ftpa1-2.fna.fbcdn.net&_nc_cat=106&_nc_oc=Q6cZ2QEaDHEI_lIFlH5iwhszW5pEEXGdpfRxVvd1fsNxComJzWTcr0b9F2IN0mSBb3D3RV4PQXMZbnbGMlQTAFjpyHKr&_nc_ohc=1Goz8zZm-UwQ7kNvwFBD8gQ&_nc_gid=X6oHi9B_XCpwaC8DUiGj1g&edm=APs17CUBAAAA&ccb=7-5&oh=00_AfuDnAfwjCc35-xPF-brrO7T2s_74l48_STZuaJI9DVorA&oe=699E589D&_nc_sid=10d13b (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x79c1d42e4470>: Failed to establish a new connection: [Errno 101] Network is unreachable'))


CLIP embedding:  94%|█████████▍| 1073/1140 [05:42<00:23,  2.84reel/s]

Download failed ig_3503362566223756411: HTTPSConnectionPool(host='instagram.fcps4-1.fna.fbcdn.net', port=443): Max retries exceeded with url: /v/t51.2885-15/466596062_18429827314078733_6433147316212954394_n.jpg?stp=dst-jpg_e15_tt6&_nc_ht=instagram.fcps4-1.fna.fbcdn.net&_nc_cat=105&_nc_oc=Q6cZ2QGVwTFAITbGZJMRZdyNcst8FCTu44-wifBQglk6rfzjP05TyKVkVoWEnh0AtxtNMSpJQ7NvWURWOV_dxoNt34yO&_nc_ohc=hIi4uxw_U1QQ7kNvwFkSv5g&_nc_gid=uEj1g-LgSTw6bTJ2tRCUFw&edm=APs17CUBAAAA&ccb=7-5&oh=00_AfsIovWWYO5sYJ68jqx9KBpnrUOIm_Q2fOXgNGG64G75lg&oe=699E504A&_nc_sid=10d13b (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x79c1d712e420>: Failed to establish a new connection: [Errno 101] Network is unreachable'))


CLIP embedding: 100%|██████████| 1140/1140 [06:02<00:00,  3.15reel/s]


Done!
   Embedded : 1,123
   Failed   : 0
   Skipped  : 17
   Image collection: 1,123 docs



In [17]:
client = chromadb.PersistentClient(path="./chroma_db")

def export_to_pkl(collection_name: str, output_path: str):
    col   = client.get_collection(collection_name)
    total = col.count()
    print(f"Exporting {collection_name}: {total:,} vectors...")

    BATCH = 500
    all_ids        = []
    all_embeddings = []
    all_documents  = []
    all_metadatas  = []

    for offset in tqdm(range(0, total, BATCH), desc=f"Pulling {collection_name}"):
        result = col.get(
            limit=BATCH,
            offset=offset,
            include=["embeddings", "documents", "metadatas"]
        )
        all_ids.extend(result["ids"])
        all_embeddings.extend(result["embeddings"])
        all_metadatas.extend(result["metadatas"])
        if result.get("documents"):
            all_documents.extend(result["documents"])

    payload = {
        "ids":        all_ids,
        "embeddings": all_embeddings,
        "metadatas":  all_metadatas,
        "documents":  all_documents,
        "count":      len(all_ids),
        "collection": collection_name,
    }

    with open(output_path, "wb") as f:
        pickle.dump(payload, f)

    size = os.path.getsize(output_path) / 1e6
    print(f"Saved {output_path} — {len(all_ids):,} vectors — {size:.1f} MB\n")
    return payload

text_data  = export_to_pkl("reel_text",   "text_embeddings.pkl")
image_data = export_to_pkl("reel_images", "image_embeddings.pkl")

print("Verification:")
print(f"   text_embeddings.pkl  -> {text_data['count']:,} vectors, dim={len(text_data['embeddings'][0])}")
print(f"   image_embeddings.pkl -> {image_data['count']:,} vectors, dim={len(image_data['embeddings'][0])}")

Exporting reel_text: 1,134 vectors...


Pulling reel_text: 100%|██████████| 3/3 [00:00<00:00, 17.93it/s]


Saved text_embeddings.pkl — 1,134 vectors — 6.1 MB

Exporting reel_images: 1,123 vectors...


Pulling reel_images: 100%|██████████| 3/3 [00:00<00:00, 18.60it/s]


Saved image_embeddings.pkl — 1,123 vectors — 6.7 MB

Verification:
   text_embeddings.pkl  -> 1,134 vectors, dim=384
   image_embeddings.pkl -> 1,123 vectors, dim=512


In [21]:
CHROMA_DB_PATH        = "./chroma_db"
PROCESSED_JSON_PATH   = "./processed.json"
TEXT_EMBEDDINGS_PKL   = "./text_embeddings.pkl"
IMAGE_EMBEDDINGS_PKL  = "./image_embeddings.pkl"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[pipeline] Device: {device.upper()}")


print("[pipeline] Loading models...")
text_model     = SentenceTransformer("all-MiniLM-L6-v2", device=device)
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("[pipeline] Models ready!")

_chroma_client   = chromadb.PersistentClient(path=CHROMA_DB_PATH)
text_collection  = _chroma_client.get_or_create_collection(
    name="reel_text",   metadata={"hnsw:space": "cosine"})
image_collection = _chroma_client.get_or_create_collection(
    name="reel_images", metadata={"hnsw:space": "cosine"})

print(f"[pipeline] Text collection : {text_collection.count()} docs")
print(f"[pipeline] Image collection: {image_collection.count()} docs")


with open(PROCESSED_JSON_PATH, encoding="utf-8") as f:
    _reels = json.load(f)

reel_lookup: dict = {str(r["id"]): r for r in _reels}
print(f"[pipeline] Loaded {len(_reels)} reels fromprocessed.json")

def _load_pkl(path: str, label: str) -> dict:
    """Normalises any pkl format to {reel_id: np.ndarray}."""
    if not os.path.exists(path):
        print(f"[pipeline] ⚠️  {label} pkl not found at {path} — dedup will be skipped")
        return {}
    with open(path, "rb") as f:
        data = pickle.load(f)

    if isinstance(data, dict):
        return {str(k): np.array(v) for k, v in data.items()}

    if isinstance(data, (list, np.ndarray)):
        arr     = np.array(data)
        id_list = [str(r["id"]) for r in _reels]
        n       = min(len(arr), len(id_list))
        return {id_list[i]: arr[i] for i in range(n)}

    print(f"[pipeline] ⚠️  Unrecognised pkl format for {label}: {type(data)}")
    return {}


text_embeds_pkl  = _load_pkl(TEXT_EMBEDDINGS_PKL,  "text")
image_embeds_pkl = _load_pkl(IMAGE_EMBEDDINGS_PKL, "image")
print(f"[pipeline] Text pkl : {len(text_embeds_pkl)} entries")
print(f"[pipeline] Image pkl: {len(image_embeds_pkl)} entries")


def build_where_filter(
    min_likes:    Optional[int] = None,
    max_duration: Optional[int] = None,
    owner:        Optional[str] = None,
    source:       Optional[str] = None,
) -> Optional[dict]:
    conditions = []
    if min_likes    is not None: conditions.append({"likes":    {"$gte": min_likes}})
    if max_duration is not None: conditions.append({"duration": {"$lte": max_duration}})
    if owner        is not None: conditions.append({"owner":    {"$eq":  owner}})
    if source       is not None: conditions.append({"source":   {"$eq":  source}})

    if not conditions:       return None
    if len(conditions) == 1: return conditions[0]
    return {"$and": conditions}



def _parse_chroma(res: dict) -> list[dict]:
    out = []
    for reel_id, dist, meta in zip(
        res["ids"][0], res["distances"][0], res["metadatas"][0]
    ):
        out.append({
            "id":       reel_id,
            "score":    round(1 - dist, 4),
            "metadata": meta,
            "reel":     reel_lookup.get(reel_id, {}),
        })
    return out


def text_search(
    query: str,
    top_k: int = 50,
    where: Optional[dict] = None,
) -> list[dict]:
    embed  = text_model.encode(query, normalize_embeddings=True).tolist()
    kwargs = dict(
        query_embeddings=[embed],
        n_results=min(top_k, text_collection.count()),
        include=["distances", "metadatas"],
    )
    if where: kwargs["where"] = where
    return _parse_chroma(text_collection.query(**kwargs))


def image_text_search(
    query: str,
    top_k: int = 50,
    where: Optional[dict] = None,
) -> list[dict]:
    if image_collection.count() == 0:
        return []
    inputs = clip_processor(text=[query], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        out   = clip_model.get_text_features(**inputs)

        raw   = out.pooler_output if hasattr(out, "pooler_output") else out
        feats = F.normalize(raw, p=2, dim=-1)
    embed  = feats[0].cpu().numpy().tolist()
    kwargs = dict(
        query_embeddings=[embed],
        n_results=min(top_k, image_collection.count()),
        include=["distances", "metadatas"],
    )
    if where: kwargs["where"] = where
    return _parse_chroma(image_collection.query(**kwargs))

def hybrid_search(
    query:       str,
    top_k:       int   = 60,
    text_weight: float = 0.6,
    where:       Optional[dict] = None,
) -> list[dict]:
    text_results  = text_search(query,       top_k=top_k, where=where)
    image_results = image_text_search(query, top_k=top_k, where=where)

    K = 60
    rrf_scores: dict[str, float] = defaultdict(float)

    for rank, r in enumerate(text_results):
        rrf_scores[r["id"]] += text_weight * (1 / (K + rank + 1))
    for rank, r in enumerate(image_results):
        rrf_scores[r["id"]] += (1 - text_weight) * (1 / (K + rank + 1))

    all_results = {r["id"]: r for r in text_results}
    for r in image_results:
        if r["id"] not in all_results:
            all_results[r["id"]] = r

    merged = []
    for reel_id, rrf_score in sorted(rrf_scores.items(), key=lambda x: -x[1]):
        entry = all_results[reel_id].copy()
        entry["rrf_score"] = round(rrf_score, 6)
        merged.append(entry)

    return merged[:top_k]

def deduplicate(
    results:            list[dict],
    text_sim_threshold: float = 0.92,
    same_owner_gap:     int   = 2,
) -> list[dict]:
    ids = [r["id"] for r in results]

    # Pass 1 — near-duplicate text removal via cosine sim on pkl embeddings
    embeds, valid_ids = [], []
    for rid in ids:
        if rid in text_embeds_pkl:
            embeds.append(text_embeds_pkl[rid])
            valid_ids.append(rid)

    dropped = set()
    if len(embeds) > 1:
        mat = np.stack(embeds)
        sim = mat @ mat.T
        for i in range(len(valid_ids)):
            if valid_ids[i] in dropped: continue
            for j in range(i + 1, len(valid_ids)):
                if sim[i, j] >= text_sim_threshold:
                    dropped.add(valid_ids[j])

    deduped = [r for r in results if r["id"] not in dropped]

    # Pass 2 — owner spreading
    final: list[dict]        = []
    delayed: list[dict]      = []
    owner_last: dict[str, int] = {}

    def flush_delayed():
        still = []
        for item in delayed:
            o = item["metadata"].get("owner", "")
            if len(final) - owner_last.get(o, -999) >= same_owner_gap:
                owner_last[o] = len(final)
                final.append(item)
            else:
                still.append(item)
        return still

    for r in deduped:
        o = r["metadata"].get("owner", "")
        delayed = flush_delayed()
        if len(final) - owner_last.get(o, -999) >= same_owner_gap:
            owner_last[o] = len(final)
            final.append(r)
        else:
            delayed.append(r)

    final.extend(delayed)

    print(f"[pipeline] Dedup: {len(results)} → {len(final)} "
          f"({len(dropped)} near-dupes removed)")
    return final



def query_reels(
    query:              str,
    top_k:              int   = 10,
    text_weight:        float = 0.6,
    min_likes:          Optional[int]   = None,
    max_duration:       Optional[int]   = None,
    owner:              Optional[str]   = None,
    source:             Optional[str]   = None,
    text_sim_threshold: float = 0.92,
    same_owner_gap:     int   = 2,
) -> list[dict]:
    """
    Full pipeline:
      1. Build metadata filter
      2. Hybrid RRF search (text + CLIP)
      3. Redundancy filter (near-dupes + owner spread)
      4. Return top_k clean results
    """
    where = build_where_filter(
        min_likes=min_likes, max_duration=max_duration,
        owner=owner, source=source,
    )

    print(f'[pipeline] Query: "{query}"' + (f" | Filter: {where}" if where else ""))

    candidates = hybrid_search(
        query, top_k=top_k * 5,
        text_weight=text_weight, where=where,
    )
    print(f"[pipeline] Hybrid candidates: {len(candidates)}")

    clean = deduplicate(
        candidates,
        text_sim_threshold=text_sim_threshold,
        same_owner_gap=same_owner_gap,
    )
    return clean[:top_k]



if __name__ == "__main__":
    results = query_reels("funny student struggles", top_k=5)
    for i, r in enumerate(results, 1):
        print(f"{i}. [{r['rrf_score']}] @{r['metadata'].get('owner','')} — "
              f"{r['metadata'].get('caption','')[:80]}")

[pipeline] Device: CPU
[pipeline] Loading models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[pipeline] Models ready!
[pipeline] Text collection : 1134 docs
[pipeline] Image collection: 1123 docs
[pipeline] Loaded 1140 reels fromprocessed.json
[pipeline] Text pkl : 6 entries
[pipeline] Image pkl: 6 entries
[pipeline] Query: "funny student struggles"
[pipeline] Hybrid candidates: 25
[pipeline] Dedup: 25 → 25 (0 near-dupes removed)
1. [0.009836] @ — 💀

@nitrchads
the ultimate meme page and ONLY sanctuary for every NIT Rourkela s
2. [0.009677] @ — 🔥🚩

@nitrchads
the ultimate meme page and ONLY sanctuary for every NIT Rourkela 
3. [0.009524] @ — Wait for end 💀

@nitrchads
the ultimate meme page and ONLY sanctuary for every N
4. [0.009375] @ — What’s up, fam? College is a wild ride, and you’re bound to encounter all sorts 
5. [0.009231] @ — ASHISH SOLANKI IS ABOUT TO HIT PECFEST! 💣🔥

Clear the stage. Brace your ribs.
Be


In [22]:
CHROMA_DB_PATH = "./chroma_db"
PROCESSED_JSON_PATH = "./processed.json"
TEXT_EMBEDDINGS_PKL = "./text_embeddings.pkl"
IMAGE_EMBEDDINGS_PKL = "./image_embeddings.pkl"

In [23]:

_client          = chromadb.PersistentClient(path=CHROMA_DB_PATH)
text_collection  = _client.get_or_create_collection("reel_text",   metadata={"hnsw:space": "cosine"})
image_collection = _client.get_or_create_collection("reel_images", metadata={"hnsw:space": "cosine"})
print(f"[pipeline] Text: {text_collection.count()} | Image: {image_collection.count()}")

with open(PROCESSED_JSON_PATH, encoding="utf-8") as f:
    _reels = json.load(f)
reel_lookup = {str(r["id"]): r for r in _reels}
print(f"[pipeline] Loaded {len(_reels)} reels")

def _load_pkl(path: str, label: str) -> dict:
    if not os.path.exists(path):
        print(f"[pipeline] ⚠️  {label} pkl not found — dedup skipped")
        return {}
    with open(path, "rb") as f:
        data = pickle.load(f)
    if isinstance(data, dict):
        return {str(k): np.array(v) for k, v in data.items()}
    if isinstance(data, (list, np.ndarray)):
        arr     = np.array(data)
        id_list = [str(r["id"]) for r in _reels]
        n       = min(len(arr), len(id_list))
        return {id_list[i]: arr[i] for i in range(n)}
    return {}

text_embeds_pkl  = _load_pkl(TEXT_EMBEDDINGS_PKL,  "text")
image_embeds_pkl = _load_pkl(IMAGE_EMBEDDINGS_PKL, "image")
print(f"[pipeline] Text pkl: {len(text_embeds_pkl)} | Image pkl: {len(image_embeds_pkl)}")

[pipeline] Text: 1134 | Image: 1123
[pipeline] Loaded 1140 reels
[pipeline] Text pkl: 6 | Image pkl: 6


In [24]:
def build_where_filter(min_likes=None, max_duration=None, owner=None, source=None):
    conditions = []

    if min_likes is not None:
        conditions.append({"likes": {"$gte": min_likes}})

    if max_duration is not None:
        conditions.append({"duration": {"$lte": max_duration}})

    if owner is not None:
        conditions.append({"owner": {"$eq": owner}})

    if source is not None:
        conditions.append({"source": {"$eq": source}})

    if not conditions:
        return None

    if len(conditions) == 1:
        return conditions[0]

    return {"$and": conditions}

def _parse_chroma(res):
    return [
        {
            "id": rid,
            "score": round(1 - dist, 4),
            "metadata": meta,
            "reel": reel_lookup.get(rid, {})
        }
        for rid, dist, meta in zip(
            res["ids"][0],
            res["distances"][0],
            res["metadatas"][0]
        )
    ]


def text_search(query, top_k=50, where=None):
    embed = text_model.encode(query, normalize_embeddings=True).tolist()

    kwargs = dict(
        query_embeddings=[embed],
        n_results=min(top_k, text_collection.count()),
        include=["distances", "metadatas"],
    )

    if where:
        kwargs["where"] = where

    return _parse_chroma(text_collection.query(**kwargs))


def image_text_search(query, top_k=50, where=None):
    if image_collection.count() == 0:
        return []

    inputs = clip_processor(
        text=[query],
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        out = clip_model.get_text_features(**inputs)
        raw = out.pooler_output if hasattr(out, "pooler_output") else out
        feats = F.normalize(raw, p=2, dim=-1)

    embed = feats[0].cpu().numpy().tolist()

    kwargs = dict(
        query_embeddings=[embed],
        n_results=min(top_k, image_collection.count()),
        include=["distances", "metadatas"],
    )

    if where:
        kwargs["where"] = where

    return _parse_chroma(image_collection.query(**kwargs))



def hybrid_search(query, top_k=60, text_weight=0.6, where=None):
    text_res = text_search(query, top_k=top_k, where=where)
    image_res = image_text_search(query, top_k=top_k, where=where)

    K = 60
    scores = defaultdict(float)

    for rank, r in enumerate(text_res):
        scores[r["id"]] += text_weight * (1 / (K + rank + 1))

    for rank, r in enumerate(image_res):
        scores[r["id"]] += (1 - text_weight) * (1 / (K + rank + 1))

    all_res = {r["id"]: r for r in text_res}

    for r in image_res:
        if r["id"] not in all_res:
            all_res[r["id"]] = r

    merged = []

    for rid, rrf in sorted(scores.items(), key=lambda x: -x[1]):
        entry = all_res[rid].copy()
        entry["rrf_score"] = round(rrf, 6)
        merged.append(entry)

    return merged[:top_k]

In [25]:
def deduplicate(results, text_sim_threshold=0.92, same_owner_gap=2):
    ids = [r["id"] for r in results]

    embeds, valid_ids = [], []
    for rid in ids:
        if rid in text_embeds_pkl:
            embeds.append(text_embeds_pkl[rid])
            valid_ids.append(rid)

    dropped = set()

    # Remove near-duplicate content using embedding similarity
    if len(embeds) > 1:
        mat = np.stack(embeds)
        sim = mat @ mat.T

        for i in range(len(valid_ids)):
            if valid_ids[i] in dropped:
                continue

            for j in range(i + 1, len(valid_ids)):
                if sim[i, j] >= text_sim_threshold:
                    dropped.add(valid_ids[j])

    deduped = [r for r in results if r["id"] not in dropped]

    final = []
    delayed = []
    owner_last = {}

    def flush():
        still = []
        for item in delayed:
            owner = item["metadata"].get("owner", "")
            if len(final) - owner_last.get(owner, -999) >= same_owner_gap:
                owner_last[owner] = len(final)
                final.append(item)
            else:
                still.append(item)
        return still

    for r in deduped:
        owner = r["metadata"].get("owner", "")
        delayed = flush()

        if len(final) - owner_last.get(owner, -999) >= same_owner_gap:
            owner_last[owner] = len(final)
            final.append(r)
        else:
            delayed.append(r)

    final.extend(delayed)

    print(
        f"[pipeline] Dedup: {len(results)} → {len(final)} "
        f"({len(dropped)} near-dupes removed)"
    )

    return final

In [28]:
def query_reels(
    query,
    top_k=10,
    text_weight=0.6,
    min_likes=None,
    max_duration=None,
    owner=None,
    source=None,
    text_sim_threshold=0.92,
    same_owner_gap=2,
):
    where = build_where_filter(
        min_likes=min_likes,
        max_duration=max_duration,
        owner=owner,
        source=source,
    )

    print(f'[pipeline] Query: "{query}"' + (f" | Filter: {where}" if where else ""))

    candidates = hybrid_search(
        query,
        top_k=top_k * 5,
        text_weight=text_weight,
        where=where,
    )

    print(f"[pipeline] Hybrid candidates: {len(candidates)}")

    return deduplicate(
        candidates,
        text_sim_threshold,
        same_owner_gap
    )[:top_k]

class IdeaStructure(BaseModel):
    concept: str
    hook: str
    structure: List[str]
    emotion: str
    why_it_works: str
    reference_url: str


class OptimizationVariant(BaseModel):
    change: str
    add: str
    result: str


class OptimizationSuggestion(BaseModel):
    second_idea_emotional_variant: OptimizationVariant


class BestFitRecommendation(BaseModel):
    best_idea_index: int
    reason: str


class AnalysisBlock(BaseModel):
    performance_drivers: List[str]
    engagement_triggers: List[str]


class StrategistOutput(BaseModel):
    analysis: AnalysisBlock
    patterns: List[str]
    ideas: List[IdeaStructure]
    best_fit_recommendation: BestFitRecommendation
    optimization_suggestion: OptimizationSuggestion



class HybridReelRetriever(BaseRetriever):
    top_k: int = Field(default=8)
    text_weight: float = Field(default=0.6)
    min_likes: Optional[int] = Field(default=None)
    max_duration: Optional[int] = Field(default=None)
    text_sim_threshold: float = Field(default=0.92)
    same_owner_gap: int = Field(default=2)

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(self, query: str) -> List[Document]:
        results = query_reels(
            query,
            top_k=self.top_k,
            text_weight=self.text_weight,
            min_likes=self.min_likes,
            max_duration=self.max_duration,
            text_sim_threshold=self.text_sim_threshold,
            same_owner_gap=self.same_owner_gap,
        )

        docs = []

        for r in results:
            meta = r["metadata"]
            reel = r.get("reel", {})

            content = (
                f"Caption: {meta.get('caption','')}\n"
                f"Transcript: {reel.get('transcript','')}"
            ).strip()

            docs.append(
                Document(
                    page_content=content,
                    metadata={
                        "id": r["id"],
                        "owner": meta.get("owner", ""),
                        "likes": meta.get("likes", 0),
                        "comments": meta.get("comments", 0),
                        "duration": meta.get("duration", 0),
                        "source": meta.get("source", ""),
                        "rrf_score": r.get("rrf_score", r.get("score", 0)),
                        "url": meta.get("embed_url", meta.get("video_url", "")),
                    },
                )
            )

        return docs

/tmp/ipython-input-490610431.py:74: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class HybridReelRetriever(BaseRetriever):


In [51]:
_base_llm = ChatGroq(
    api_key="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",
    model="llama-3.3-70b-versatile",
    temperature=0.85,
    max_tokens=2048,
)


system_prompt = """
You are an expert social media performance analyst and viral content strategist specializing in Instagram Reels.
You analyze high-performing reels to uncover repeatable success patterns and transform them into highly actionable viral content ideas.

Focus on:
• audience psychology  • scroll-stopping hooks  • retention mechanics
• emotional triggers   • relatability & shareability

Avoid generic observations.

You are given REAL reels retrieved from a database. Each reel may include:
- captions, transcript excerpts, engagement metrics, creator handles

━━━━━━━━━━━━━━━━━━━━
YOUR TASK
━━━━━━━━━━━━━━━━━━━━
1️⃣ Analyze why the retrieved reels performed well (hook effectiveness, curiosity gaps,
   pacing, emotional triggers, novelty vs familiarity). Be specific and concise.

2️⃣ Identify repeating patterns (hook formats, storytelling flow, tone, pain points,
   visual framing, engagement triggers). Prioritize: repeatable, psychologically
   compelling, niche-relevant patterns.

3️⃣ Generate 3 HIGHLY SPECIFIC viral reel ideas from those patterns.
   Native to the niche, optimized for retention and shares. No generic trends.

━━━━━━━━━━━━━━━━━━━━
QUALITY RULES
━━━━━━━━━━━━━━━━━━━━
• Specific not generic
• Optimize retention & shareability
• Hooks must create curiosity gaps
• Ideas must be immediately executable
• Favor clarity over cleverness

━━━━━━━━━━━━━━━━━━━━
STRICT OUTPUT FORMAT
━━━━━━━━━━━━━━━━━━━━
Respond with ONLY a valid JSON object — no markdown, no explanation, no extra text.

{{
  "analysis": {{
    "performance_drivers": ["...", "..."],
    "engagement_triggers": ["...", "..."]
  }},
  "patterns": ["...", "..."],
  "ideas": [
    {{
      "concept": "...",
      "hook": "...",
      "structure": ["step 1", "step 2", "step 3", "step 4", "step 5"],
      "emotion": "...",
      "why_it_works": "...",
      "reference_url": "..."
    }}
  ],
  "best_fit_recommendation": {{
    "best_idea_index": 0,
    "reason": "..."
  }},
  "optimization_suggestion": {{
    "second_idea_emotional_variant": {{
      "change": "...",
      "add": "...",
      "result": "..."
    }}
  }}
}}

Retrieved Reels:
{context}
"""


prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [52]:
retriever    = HybridReelRetriever(top_k=8, text_weight=0.6)
chat_history = []

def conversational_rag(query: str) -> dict:

    docs = retriever._get_relevant_documents(query)


    context = "\n\n---\n\n".join([
        f"Owner: @{d.metadata.get('owner','?')} | "
        f"Likes: {d.metadata.get('likes',0):,} | "
        f"Duration: {d.metadata.get('duration',0)}s\n{d.page_content}"
        for d in docs
    ])


    messages = [("system", system_prompt.replace("{context}", context))]
    messages += [(m.type, m.content) for m in chat_history]
    messages += [("human", query)]


    raw = _base_llm.invoke(messages).content


    try:
        clean  = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        parsed = StrategistOutput.model_validate_json(clean)
    except Exception as e:
        print(f"[parse error] {e}\nRaw:\n{raw[:500]}")
        parsed = None

    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=raw))

    return {"answer": parsed, "context": docs}


def chat_with_analyst(queries: list):
    print("\n💬 Starting strategist session...\n")

    for query in queries:
        print(f"\n👤 You: {query}\n")

        result = conversational_rag(query)
        answer: StrategistOutput = result["answer"]

        if answer is None:
            print("⚠️  Could not parse LLM output — see raw above")
            continue

        print("🤖 Strategist Output:")
        print(json.dumps(answer.model_dump(), indent=2, ensure_ascii=False))

        print("\n📦 Sources used:")
        for doc in result["context"]:
            meta = doc.metadata
            print(
                f"  @{meta.get('owner','?')} | ❤️ {meta.get('likes',0):,} | "
                f"⏱ {meta.get('duration',0)}s | score: {meta.get('rrf_score','—')} | "
                f"{meta.get('url','')}"
            )

        print("\n" + "=" * 70)


chat_with_analyst([
    "funny relatable student struggles",
    "make the second idea more emotional and less funny",
    "which idea works best for a college page with 10k followers?"
])


💬 Starting strategist session...


👤 You: funny relatable student struggles

[pipeline] Query: "funny relatable student struggles"
[pipeline] Hybrid candidates: 40
[pipeline] Dedup: 40 → 40 (0 near-dupes removed)
🤖 Strategist Output:
{
  "analysis": {
    "performance_drivers": [
      "relatability of student struggles",
      "humor and light-hearted tone"
    ],
    "engagement_triggers": [
      "nods of recognition from students",
      "shared experiences and emotions"
    ]
  },
  "patterns": [
    "using humor to cope with academic stress",
    "highlighting quirks of college life",
    "poking fun at student stereotypes"
  ],
  "ideas": [
    {
      "concept": "Morning Routine Struggles",
      "hook": "Waking up for an 8am class",
      "structure": [
        "step 1: introduction to the struggle",
        "step 2: exaggerated morning routine",
        "step 3: comedic twist on typical morning habits",
        "step 4: punchline or unexpected turn",
        "step 5: call-to

In [53]:
from __future__ import annotations

import json
import numpy as np
from typing import Optional, List, Any
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, END


class PipelineState(TypedDict):
    query: str
    expanded_query: str
    retrieved_docs: List[Any]
    reranked_docs: List[Any]
    strategist_output: Optional[Any]
    eval_passed: bool
    retry_count: int
    final_output: Optional[dict]
    alpha: float


MAX_RETRIES          = 2
TOP_K                 = 8
DEFAULT_ALPHA        = 0.4
MIN_IDEAS_NEEDED     = 2
MIN_IDEA_CONCEPT_LEN = 10


def retrieve_node(state: PipelineState) -> PipelineState:
    query = state["expanded_query"] if state["retry_count"] > 0 else state["query"]
    print(f"\n[LangGraph | retrieve] Query: '{query}' | Retry #{state['retry_count']}")
    docs = query_reels(query, top_k=TOP_K * 3, text_weight=0.6)
    return {**state, "retrieved_docs": docs}


def cosine_rerank_node(state: PipelineState) -> PipelineState:
    docs  = state["retrieved_docs"]
    query = state["expanded_query"] if state["retry_count"] > 0 else state["query"]
    alpha = state.get("alpha", DEFAULT_ALPHA)

    if not docs:
        return {**state, "reranked_docs": docs}

    print(f"[LangGraph | rerank] Reranking {len(docs)} docs (alpha={alpha})")

    query_vec = text_model.encode(query, normalize_embeddings=True)

    reranked = []
    for doc in docs:
        meta     = doc.get("metadata", {})
        reel     = doc.get("reel", {})
        doc_text = f"{meta.get('caption', '')} {reel.get('transcript', '')}".strip()

        cosine_sim    = float(np.dot(query_vec, text_model.encode(doc_text, normalize_embeddings=True))) if doc_text else 0.0
        rrf_score     = doc.get("score", 0.0)
        blended_score = alpha * cosine_sim + (1 - alpha) * rrf_score

        reranked.append({**doc, "blended_score": round(blended_score, 5)})

    reranked.sort(key=lambda x: x["blended_score"], reverse=True)
    reranked = reranked[:TOP_K]

    print(f"[LangGraph | rerank] Top doc score: {reranked[0]['blended_score']} | Bottom: {reranked[-1]['blended_score']}")
    return {**state, "reranked_docs": reranked}


def strategist_node(state: PipelineState) -> PipelineState:
    docs  = state["reranked_docs"]
    query = state["query"]
    retry = state["retry_count"]

    print(f"[LangGraph | strategist] Running LLM on {len(docs)} docs...")

    # ── FIXED: d["metadata"] not d.metadata ──────────────────────────────────
    context = "\n\n---\n\n".join([
        f"Owner: @{d['metadata'].get('owner','?')} | "
        f"Likes: {d['metadata'].get('likes', 0):,} | "
        f"Duration: {d['metadata'].get('duration', 0)}s | "
        f"URL: {d['metadata'].get('embed_url', d['metadata'].get('video_url',''))}\n"
        f"Caption: {d['metadata'].get('caption', '')}\n"
        f"Transcript: {d.get('reel', {}).get('transcript', '')}"
        for d in docs
    ])

    retry_nudge = ""
    if retry > 0:
        retry_nudge = (
            "\n\n⚠️ RETRY INSTRUCTION: Previous output was too generic. "
            "Ideas must be hyper-specific to the niche, not broad templates. "
            "Use concrete details from the retrieved reels above."
        )

    filled_prompt = system_prompt.replace("{context}", context) + retry_nudge

    messages  = [("system", filled_prompt)]
    messages += [(m.type, m.content) for m in chat_history]
    messages += [("human", query)]

    time.sleep(5) # Add a delay to avoid rate limiting
    raw = _base_llm.invoke(messages).content

    try:
        clean  = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        parsed = StrategistOutput.model_validate_json(clean)
    except Exception as e:
        print(f"[LangGraph | strategist] Parse error: {e}")
        parsed = None

    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=raw))

    return {**state, "strategist_output": parsed}


def evaluator_node(state: PipelineState) -> PipelineState:
    output = state["strategist_output"]

    if output is None:
        print("[LangGraph | evaluator] ❌ No parsed output — triggering retry")
        return {**state, "eval_passed": False}

    ideas       = output.ideas
    num_ideas   = len(ideas)
    short_hooks = sum(1 for i in ideas if len(i.concept) < MIN_IDEA_CONCEPT_LEN)
    has_best    = output.best_fit_recommendation is not None

    passed = num_ideas >= MIN_IDEAS_NEEDED and short_hooks == 0 and has_best

    reason = []
    if num_ideas < MIN_IDEAS_NEEDED: reason.append(f"only {num_ideas} ideas (need {MIN_IDEAS_NEEDED})")
    if short_hooks > 0:              reason.append(f"{short_hooks} concepts too short/generic")
    if not has_best:                 reason.append("missing best_fit_recommendation")

    if passed:
        print(f"[LangGraph | evaluator] ✅ Output passed ({num_ideas} ideas, all specific)")
    else:
        print(f"[LangGraph | evaluator] ❌ Failed: {', '.join(reason)}")

    return {**state, "eval_passed": passed}


def query_expander_node(state: PipelineState) -> PipelineState:
    query = state["query"]
    print(f"[LangGraph | expander] Broadening query for retry...")

    expand_prompt = (
        f"The query '{query}' returned low-quality results. "
        "Rewrite it as a slightly broader, more general version (5-10 words max). "
        "Return ONLY the new query string, nothing else."
    )

    expanded = _base_llm.invoke([("human", expand_prompt)]).content.strip().strip('"').strip("'")
    print(f"[LangGraph | expander] '{query}' → '{expanded}'")

    return {**state, "expanded_query": expanded, "retry_count": state["retry_count"] + 1}


def finalizer_node(state: PipelineState) -> PipelineState:
    output = state["strategist_output"]
    docs   = state["reranked_docs"]

    final = {
        "query":        state["query"],
        "retries_used": state["retry_count"],
        "alpha_used":   state.get("alpha", DEFAULT_ALPHA),
        "answer":       output.model_dump() if output else None,
        "sources": [
    {
        "owner":    d["metadata"].get("owner", "?"),
        "likes":    d["metadata"].get("likes", 0),
        "duration": d["metadata"].get("duration", 0),
        "score":    d.get("blended_score", d.get("score", 0)),
        "url":      d["metadata"].get("embed_url") or d["metadata"].get("video_url", ""),
    }
    for d in docs
],
    }

    print(f"\n[LangGraph | final] Done. Retries used: {state['retry_count']}")
    return {**state, "final_output": final}


def route_after_eval(state: PipelineState) -> str:
    if state["eval_passed"]:
        return "finalize"
    if state["retry_count"] < MAX_RETRIES:
        return "expand_and_retry"
    print("[LangGraph | router] Max retries reached — finalizing anyway")
    return "finalize"


def build_graph():
    g = StateGraph(PipelineState)

    g.add_node("retrieve",   retrieve_node)
    g.add_node("rerank",     cosine_rerank_node)
    g.add_node("strategist", strategist_node)
    g.add_node("evaluator",  evaluator_node)
    g.add_node("expander",   query_expander_node)
    g.add_node("finalize",   finalizer_node)

    g.set_entry_point("retrieve")
    g.add_edge("retrieve",   "rerank")
    g.add_edge("rerank",     "strategist")
    g.add_edge("strategist", "evaluator")

    g.add_conditional_edges(
        "evaluator",
        route_after_eval,
        {"finalize": "finalize", "expand_and_retry": "expander"}
    )

    g.add_edge("expander", "retrieve")
    g.add_edge("finalize", END)

    return g.compile()


pipeline_graph = build_graph()


def run_optimized_pipeline(query: str, alpha: float = DEFAULT_ALPHA) -> dict:
    initial_state: PipelineState = {
        "query":             query,
        "expanded_query":    query,
        "retrieved_docs":    [],
        "reranked_docs":     [],
        "strategist_output": None,
        "eval_passed":       False,
        "retry_count":       0,
        "final_output":      None,
        "alpha":             alpha,
    }

    final_state = pipeline_graph.invoke(initial_state)
    result      = final_state["final_output"]

    print(f"\n👤 Query: {result['query']}")
    print(f"\n🤖 Strategist Output:")
    print(json.dumps(result["answer"], indent=2, ensure_ascii=False))
    print(f"\n📦 Sources used ({len(result['sources'])}):")
    for s in result["sources"]:
        print(f"  @{s['owner']} | ❤️ {s['likes']:,} | ⏱ {s['duration']}s | score: {s['score']} | {s['url']}")
    print(f"\n⚙️  Alpha: {result['alpha_used']} | Retries: {result['retries_used']}")
    print("\n" + "=" * 70)

    return result


def run_optimized_session(queries: list, alpha: float = DEFAULT_ALPHA):
    print("\n💬 Starting optimized strategist session...\n")
    results = []
    for q in queries:
        results.append(run_optimized_pipeline(q, alpha=alpha))
    return results

In [54]:
run_optimized_session(
    [
        "funny relatable student struggles",
        "make the second idea more emotional and less funny",
        "which idea works best for a college page with 10k followers?",
    ]
)


💬 Starting optimized strategist session...


[LangGraph | retrieve] Query: 'funny relatable student struggles' | Retry #0
[pipeline] Query: "funny relatable student struggles"
[pipeline] Hybrid candidates: 120
[pipeline] Dedup: 120 → 120 (0 near-dupes removed)
[LangGraph | rerank] Reranking 24 docs (alpha=0.4)
[LangGraph | rerank] Top doc score: 0.51223 | Bottom: 0.40424
[LangGraph | strategist] Running LLM on 8 docs...
[LangGraph | evaluator] ✅ Output passed (3 ideas, all specific)

[LangGraph | final] Done. Retries used: 0

👤 Query: funny relatable student struggles

🤖 Strategist Output:
{
  "analysis": {
    "performance_drivers": [
      "relatability of student struggles",
      "humor and light-hearted tone"
    ],
    "engagement_triggers": [
      "nods of recognition from students",
      "shared experiences and emotions"
    ]
  },
  "patterns": [
    "using humor to cope with academic stress",
    "highlighting quirks of college life",
    "poking fun at student stereotypes

[{'query': 'funny relatable student struggles',
  'retries_used': 0,
  'alpha_used': 0.4,
  'answer': {'analysis': {'performance_drivers': ['relatability of student struggles',
     'humor and light-hearted tone'],
    'engagement_triggers': ['nods of recognition from students',
     'shared experiences and emotions']},
   'patterns': ['using humor to cope with academic stress',
    'highlighting quirks of college life',
    'poking fun at student stereotypes'],
   'ideas': [{'concept': 'Procrastination Nation',
     'hook': 'When you finally start studying... 10 minutes before the exam',
     'structure': ['step 1: introduction to the struggle',
      'step 2: comedic portrayal of procrastination habits',
      'step 3: exaggerated excuses and justifications',
      'step 4: punchline or unexpected turn',
      'step 5: call-to-action or relatable question'],
     'emotion': 'amusement and recognition',
     'why_it_works': "resonates with students' shared experiences and offers a lig